In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import hashlib
import osmnx as ox
from shapely import wkt

In [ ]:
# Load the dataset

##MethaneSAT Scenes
msat_scenes = gpd.read_file("msat_all_scenes.gpkg")


##Read in the geopackages
##Replace with relative filepaths to each file
osm_ordos_industrial12 = gpd.read_file("data/ordos12_dissolved.gpkg")
osm_ordos_industrial3 = gpd.read_file("data/ordos3_dissolved.gpkg")
osm_northchina_industrial = gpd.read_file("data/NorthChina_OSM_Dissolved.gpkg")
songliao_pumpjacks = gpd.read_file('songliao_pumpjacks_centroids.gpkg')
songliao_facilities = gpd.read_file("songliao_facilities_polygon.gpkg")




In [4]:
osm_tarim1 = pd.read_excel('tarim1_osm.xlsx')
osm_tarim1['geometry']=osm_tarim1['geometry'].apply(wkt.loads)
osm_tarim1 = gpd.GeoDataFrame(osm_tarim1, geometry='geometry', crs="epsg:4326")

osm_tarim2 = pd.read_excel('tarim2_osm.xlsx')
osm_tarim2['geometry']=osm_tarim2['geometry'].apply(wkt.loads)
osm_tarim2 = gpd.GeoDataFrame(osm_tarim2, geometry='geometry', crs="epsg:4326")

osm_tarim3 = pd.read_excel('tarim3_osm.xlsx')
osm_tarim3['geometry']=osm_tarim3['geometry'].apply(wkt.loads)
osm_tarim3 = gpd.GeoDataFrame(osm_tarim3, geometry='geometry', crs="epsg:4326")

junggar = pd.read_excel('junggar.xlsx')
junggar['geometry']=junggar['geometry'].apply(wkt.loads)
junggar = gpd.GeoDataFrame(junggar, geometry='geometry', crs="epsg:4326")

In [ ]:
##Selecting specific methaneSAT scenes
##Returns geodataframe with only 1 polygon, the scene of interest

ordos1 = msat_scenes[msat_scenes['name']=="ORDOS: 1"]
ordos2 = msat_scenes[msat_scenes['name']=="ORDOS: 2"]
tarim1 = msat_scenes[msat_scenes['name']=="TARIM: 1"]
tarim2 = msat_scenes[msat_scenes['name']=="TARIM: 2"]
tarim3 = msat_scenes[msat_scenes['name']=="TARIM: 3"]
junggar = msat_scenes[msat_scenes['name']=="JUNGGAR: 1"]
sichuan1 = msat_scenes[msat_scenes['name']=="SICHUAN: 1"]
sichuan2 = msat_scenes[msat_scenes['name']=="SICHUAN: 2"]
sichuan3 = msat_scenes[msat_scenes['name']=="SICHUAN: 3"]
sichuan4 = msat_scenes[msat_scenes['name']=="SICHUAN: 4"]

#print(msat_scenes.crs)


In [ ]:
# OSM Query


##dictionary of tags for query, can be modified
tags = {
    'building': 'industrial',
    'landuse':'industrial',
    'man_made':'works'
} 

##Selecting arbitrary geography to run OSM query

polygon = junggar['geometry'].unary_union

junggar_industrial = ox.features_from_polygon(polygon, tags)


#writing to both a geopackage and xlsx file
junggar_industrial.to_file("junggar.gpkg", driver="GPKG")
junggar_industrial.to_excel('junggar.xlsx')


In [20]:
def process_geodataframe(gdf, crs_epsg, source_name, feature_type, location):
    """
    Projects a GeoDataFrame to a specific UTM CRS, calculates area in km2,
    adds lat/lon centroids, and sets Source/Type attributes.
    """
    # Create a copy to avoid warnings
    gdf_processed = gdf.copy()
    
    #Project to target CRS (with meters as unit to avoid getting area in degrees)
    gdf_processed = gdf_processed.to_crs(crs_epsg)
    
    # Calculate area in km2
    gdf_processed['area_calc_km2'] = gdf_processed['geometry'].area / 1000000
    
    # Calculate centroids (lat/lon) in WGS84 
    centroids = gdf_processed.centroid.to_crs(epsg=4326)
    gdf_processed['lat'] = centroids.y
    gdf_processed['lon'] = centroids.x
    
    # Project back to WGS84 for consistent output
    gdf_processed = gdf_processed.to_crs(epsg=4326)
    
    # Add data sources and types
    gdf_processed['Source'] = source_name #How data was sourced
    gdf_processed['Type'] = feature_type #type of feature
    gdf_processed['location'] = location #msat scene

    return gdf_processed

In [ ]:
#Running process gdf function on OSM data acquired through previous queries


osm_ordos_industrial12 = process_geodataframe(osm_ordos_industrial12, 'EPSG:32649', "OSM", "Facility", 'Ordos')

osm_ordos_industrial3 = process_geodataframe(osm_ordos_industrial3, 'EPSG:32649', "OSM", "Facility", 'Ordos')

osm_northchina_industrial = process_geodataframe(osm_northchina_industrial, 'EPSG:32650', "OSM", "Facility", 'North China')

songliao_facilities = process_geodataframe(songliao_facilities, 'EPSG:32651', "Manual", "Facility", 'Songliao')

osm_tarim1 = process_geodataframe(osm_tarim1, 'EPSG:32644', "OSM", "Facility", 'Tarim')

osm_tarim2 = process_geodataframe(osm_tarim2, 'EPSG:32644', "OSM", "Facility", 'Tarim')

osm_tarim3 = process_geodataframe(osm_tarim3, 'EPSG:32644', "OSM", "Facility", 'Tarim')

junggar = process_geodataframe(junggar, 'EPSG:32645', "OSM", "Facility", 'Junggar')
 

In [ ]:
#making sure all gdfs are in the same CRS for join
print(songliao_pumpjacks.crs)
print(songliao_facilities.crs)
print(osm_northchina_industrial.crs)
print(osm_ordos_industrial3.crs)
print(osm_ordos_industrial12.crs)

EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326


In [ ]:
#Joining all gdfs and saving to xlsx

osm_tarim1_and_2 = pd.merge(osm_tarim1, osm_tarim2, how='outer')
osm_all_tarim = pd.merge(osm_tarim1_and_2, osm_tarim3, how='outer')

ordos_combined = pd.merge(osm_ordos_industrial3, osm_ordos_industrial12, how='outer')
ordos_and_nc = pd.merge(ordos_combined, osm_northchina_industrial, how='outer')
ordos_and_nc_and_songliao = pd.merge(ordos_and_nc, songliao_facilities, how='outer')
ordos_and_nc_and_songliao_junggar = pd.merge(ordos_and_nc_and_songliao, junggar, how='outer')

all_merged = pd.merge(ordos_and_nc_and_songliao_junggar, osm_all_tarim, how='outer')

final_dataset = all_merged[['name', 'area_calc_km2', 'lat', 'lon', 'Type', 'Source', 'osm_id', 'location']]


final_dataset.to_excel('processed_osm.xlsx')

/var/folders/25/97xyvmqx10x2d7d5k45rpb780000gp/T/ipykernel_43240/2418312399.py:6: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  ordos_and_nc_and_songliao = pd.merge(ordos_and_nc, songliao_facilities, how='outer')


In [ ]:
#summary stats of final dataset
final_dataset.groupby('location').agg({'Type': 'count', 'area_calc_km2': 'sum', 'name': 'count'})

,Type,area_calc_km2,name
location,,,
Junggar,320,88.171973,7
North China,11236,2130.694482,0
Ordos,2660,618.709422,1120
Songliao,69,5.388831,0
Tarim,861,181.655603,30
